Section A - Interview Theory

Q1. What's the difference between a Series and a DataFrame? Give one operation that works on both, and one that works only on a DataFrame.
A Series is a one-dimensional array representing a single column of data, while a DataFrame is a two-dimensional, tabular data structure composed of multiple Series. The .head() operation works on both to preview data, whereas .merge() or .info() works only on a DataFrame.

Q2. What does inplace=True do? Why is the pandas team discouraging its use in newer versions? Give one example where you'd avoid it.
inplace=True modifies the existing object directly in memory rather than returning a new copy. The pandas team discourages it because it rarely provides actual performance benefits and frequently causes unexpected behavior with chained indexing. You should avoid it when you want to keep the original raw DataFrame intact while creating a cleaned subset for analysis.

Q3. Explain the difference between .fillna() and .dropna(). When would a data analyst pick one over the other? Be specific - give a concrete situation for each.
.fillna() replaces missing values with a specified value, while .dropna() completely removes rows or columns containing missing data. An analyst would use .fillna() to replace missing ages with the median age to preserve rows, but use .dropna() to remove rows where the critical target variable (like 'churn_status') is missing entirely.

Q4. What does pd.to_numeric(col, errors='coerce') do that plain pd.to_numeric(col) does NOT? Which one would you use on real-world messy data, and why?
Adding errors='coerce' forces any unparseable strings (like "N/A" or currency symbols) to become NaN instead of throwing a script-stopping error. You would almost always use errors='coerce' on real-world messy data to prevent the entire pipeline from breaking over a single typo.

Q5. Explain .loc vs .iloc with one example each. Then describe a case where df.loc[0] and df.iloc[0] return different rows.
.loc accesses rows and columns by their explicit index labels (e.g., df.loc['row_label']), while .iloc accesses them by integer position (e.g., df.iloc[0] for the first row). If you sort or filter a DataFrame so its index becomes [5, 2, 0], df.iloc[0] returns the first visible row (index 5), but df.loc[0] returns the row specifically labeled 0 (the third visible row).

Q6. What is the difference between .map(), .apply(), and .replace() for a Series? When would you reach for each?
.replace() is used for exact 1-to-1 value substitutions (e.g., fixing typos). .map() is best for dictionary lookups mapping every value in a Series to a new value. .apply() is used when you need to pass each element through a complex custom function or lambda expression that basic mapping can't handle.

Q7. What is the SettingWithCopyWarning? Give one example of code that triggers it and one way to fix it.
This warning occurs when you try to modify a DataFrame that was created as a slice of another DataFrame, leaving pandas unsure if it should modify the original or just the copy. It is triggered by chained assignment like df[df['age'] > 20]['status'] = 'Adult'. Fix it by using .loc for assignment: df.loc[df['age'] > 20, 'status'] = 'Adult'.

Q8. Explain pd.cut() vs pd.qcut(). If you have annual income data and want to label customers as Low / Mid / High earners, which one fits better? Why?
pd.cut() divides data into bins of equal absolute numerical width, while pd.qcut() divides data into quantiles so each bin has the same number of rows. For income data, pd.qcut() fits better because income is typically highly skewed; pd.cut() would put almost everyone in the "Low" bin and only a few outliers in "High," whereas pd.qcut() evenly distributes the customer base.

Section B - Predict the Output

In [1]:
import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
df2 = df
df2['a'] = [10, 20, 30]
print(df)

    a  b
0  10  4
1  20  5
2  30  6


In [2]:
import pandas as pd
import numpy as np
s = pd.Series([1, 2, np.nan, 4, 5])
print(s.mean())
print(s.sum())

3.0
12.0


In [3]:
import pandas as pd
df = pd.DataFrame({'price': ['100', '200', 'NA', '400']})
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print(df['price'].dtype)
print(df['price'].sum())

float64
700.0


In [4]:
import pandas as pd
df = pd.DataFrame({'rating': [4.5, 3.8, 4.2, 5.0]})
df_filtered = df[df['rating'] > 4]
df_filtered['rating'] = df_filtered['rating'] * 2
print(df)
print(df_filtered)

   rating
0     4.5
1     3.8
2     4.2
3     5.0
   rating
0     9.0
2     8.4
3    10.0


/tmp/ipykernel_2045/445987763.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['rating'] = df_filtered['rating'] * 2


In [5]:
import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3, 4]})
result = df['a'].apply(lambda x: x**2 if x > 2 else 'small')
print(result)
print(result.dtype)

0    small
1    small
2        9
3       16
Name: a, dtype: object
object


In [6]:
import pandas as pd
df = pd.DataFrame({'city': ['Delhi', 'Mumbai', 'Delhi', 'Bengaluru', None]})
print(df['city'].nunique())
print(len(df['city'].unique()))
print(df['city'].value_counts())

3
4
city
Delhi        2
Mumbai       1
Bengaluru    1
Name: count, dtype: int64


In [7]:
import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3], 'b': [10, 20, 30]})
print(df[df['a'] > 1])
# print(df[df['a'] > 1 & df['b'] > 15])
print(df[(df['a'] > 1) & (df['b'] > 15)])

   a   b
1  2  20
2  3  30
   a   b
1  2  20
2  3  30


Section C - Applied on Indian Startups Dataset

In [8]:
import pandas as pd
import numpy as np


df = pd.read_csv('indian_startups_funding.csv')

df.columns = ['startup_name', 'city', 'industry', 'funding_stage', 'amount', 'funding_date', 'investors', 'founded_year']

df['startup_name'] = df['startup_name'].astype(str).str.strip().str.title()
df['city'] = df['city'].astype(str).str.strip().str.title()

city_aliases = {
    'Bangalore': 'Bengaluru',
    'Blr': 'Bengaluru',
    'Bombay': 'Mumbai',
    'Gurgaon': 'Gurugram',
    'Delhi': 'New Delhi',
    'Calcutta': 'Kolkata',
    'Madras': 'Chennai'
}
df['city'] = df['city'].replace(city_aliases)

industry_variants = {'E-Commerce': 'Ecommerce', 'Fin-Tech': 'Fintech', 'Ed-Tech': 'Edtech'}
stage_variants = {'Seed Funding': 'Seed', 'Pre Series A': 'Pre-Series A'}
df['industry'] = df['industry'].replace(industry_variants)
df['funding_stage'] = df['funding_stage'].replace(stage_variants)

df['amount'] = df['amount'].astype(str).str.replace('INR', '', regex=False)\
                                        .str.replace('Cr', '', regex=False)\
                                        .str.replace('Crore', '', regex=False)\
                                        .str.replace(',', '', regex=False)\
                                        .str.strip()
df['amount'] = df['amount'].replace('Undisclosed', np.nan)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

df['funding_date'] = pd.to_datetime(df['funding_date'], errors='coerce', dayfirst=True)

df = df.drop_duplicates()

In [9]:
import pandas as pd
raw_df = pd.read_csv('indian_startups_funding.csv')

print("(a) Total rows and columns:", raw_df.shape)
print("\n(b) Missing values per column:\n", raw_df.isna().sum())
print("\n(c) Exact duplicate rows:", raw_df.duplicated().sum())

(a) Total rows and columns: (468, 8)

(b) Missing values per column:
 Startup Name      0
City              0
INDUSTRY          0
Funding Stage     0
Amount            0
Funding Date     19
Investors        10
Founded Year     23
dtype: int64

(c) Exact duplicate rows: 18


In [10]:
sorted_columns = sorted(df.columns.tolist())
print("Cleaned column names (sorted):", sorted_columns)

Cleaned column names (sorted): ['amount', 'city', 'founded_year', 'funding_date', 'funding_stage', 'industry', 'investors', 'startup_name']


In [11]:
valid_years = df['founded_year'].dropna()
stats_dict = {
    'count': valid_years.count(),
    'mean': valid_years.mean(),
    'min': valid_years.min(),
    'max': valid_years.max(),
    'median': valid_years.median(),
    'std': valid_years.std()
}
print(stats_dict)

{'count': np.int64(427), 'mean': np.float64(2016.4285714285713), 'min': 2008.0, 'max': 2022.0, 'median': 2017.0, 'std': 4.246117096325488}


Cleaning Verification

In [12]:
sorted_cities = sorted(df['city'].dropna().unique().tolist())
print("Unique cities:", sorted_cities)

Unique cities: ['Ahmedabad', 'Bengaluru', 'Bom', 'Chennai', 'Ggn', 'Gurugram', 'Hyd', 'Hyderabad', 'Kolkata', 'Mumbai', 'New Delhi', 'Noida', 'Pune']


In [13]:
print(df['industry'].value_counts())

industry
AI/ML                22
TRAVELTECH           16
saas                 14
logistics            14
PROPTECH             13
Ecommerce            13
Travel-Tech          13
BEAUTYTECH           13
BeautyTech           12
Agri Tech            12
AI-ML                11
AIML                 11
Edtech               11
Electric Vehicles    11
Prop-Tech            11
GAMING               11
FoodTech             10
AI ML                10
Saas                 10
Health-Tech           9
Food-Tech             9
Beauty-Tech           9
Gaming                9
Food Tech             8
ecommerce             8
PropTech              8
LOGISTICS             8
Games                 8
Fintech               8
EdTech                7
foodtech              7
HealthTech            7
healthtech            6
gaming                6
EV                    6
SAAS                  6
SaaS                  6
Electric-Vehicles     5
FINTECH               5
ev                    5
Logistics             5
FOOD-TE

In [14]:
print("(a) dtype of amount column:", df['amount'].dtype)
print("(b) NaN values in amount:", df['amount'].isna().sum())
print("(c) Mean funding amount:", df['amount'].mean())
print("(d) Minimum funding:", df['amount'].min(), "| Maximum funding:", df['amount'].max())

(a) dtype of amount column: float64
(b) NaN values in amount: 281
(c) Mean funding amount: 761.2781065088758
(d) Minimum funding: 8.0 | Maximum funding: 3545.0


In [15]:
yearly_deals = df['funding_date'].dt.year.value_counts()
print("(a) Deals per year:\n", yearly_deals)
print("\n(b) Year with highest deals:", yearly_deals.idxmax())

(a) Deals per year:
 funding_date
2022.0    23
2023.0    14
2021.0    14
2024.0     7
2019.0     7
2020.0     6
2017.0     5
2018.0     5
2016.0     4
2015.0     3
2011.0     1
2014.0     1
2013.0     1
2012.0     1
Name: count, dtype: int64

(b) Year with highest deals: 2022.0


In [16]:
print(df['funding_stage'].value_counts())

funding_stage
Seed            21
Pre-Series A    20
ipo             18
Series-E        17
bridge          16
IPO             15
Bridge          15
SERIES A        14
series_b        14
SERIES C        14
Pre-Seed        14
SEED            13
seed            13
SERIES E        13
Pre Seed        12
series b        12
Series-D        12
Series-B        12
Pre-IPO         12
Series-C        12
PRE-SEED        12
Series E        11
series d        11
Series-A        10
Series D        10
Series C        10
series c        10
BRIDGE          10
Preseed         10
Pre-A            9
Series B         9
SERIES B         9
series_a         9
PRE-SERIES A     9
Series A         7
series e         7
SERIES D         5
series a         3
Name: count, dtype: int64


Filtering

In [17]:
q25_df = df[(df['city'] == 'Bengaluru') & (df['amount'] > 100)]
q25_result = q25_df[['startup_name', 'amount']].sort_values(by='amount', ascending=False)
print(q25_result)

       startup_name  amount
311        Routemap  3528.0
226     Agriconnect  3526.0
154        Ecomotor  2244.0
98           Meesho  2231.0
146       Curefoods  2230.0
252  Playtime Games  1410.0
369            Cred   950.0
67      Simplilearn   923.0
447        Agroplus   726.0
341          Practo   486.0
88            Zepto   455.0
149     Mediconnect   304.0
182         Fetchit   150.0
398         Cuemath   115.0


In [18]:
target_industries = ['Fintech', 'Edtech', 'Health Tech']
q26_count = df[df['industry'].isin(target_industries)].shape[0]
print("Total count of startups in Fintech, Edtech, or Health Tech:", q26_count)

Total count of startups in Fintech, Edtech, or Health Tech: 23


In [19]:
q27_count = df['founded_year'].between(2015, 2020, inclusive='both').sum()
print("Startups founded between 2015 and 2020:", q27_count)

Startups founded between 2015 and 2020: 211


In [20]:
pay_startups = df[df['startup_name'].str.contains('pay', case=False, na=False)]
print("Startups containing 'pay':\n", pay_startups['startup_name'].unique())

Startups containing 'pay':
 ['Rupaypro' 'Razorpay' 'Paytm' 'Paybuddy' 'Quickpay India']


Transformation & Derived Columns

In [21]:
df['funding_efficiency'] = df['amount'] / (2024 - df['founded_year'])
q29_result = df.sort_values(by='funding_efficiency', ascending=False).head(10)
print(q29_result[['startup_name', 'industry', 'amount', 'founded_year', 'funding_efficiency']])

    startup_name           industry  amount  founded_year  funding_efficiency
322   Classroomx              AI/ML  3520.0        2022.0              1760.0
154     Ecomotor             gaming  2244.0        2022.0              1122.0
158   Makemytrip  Electric Vehicles  2202.0        2022.0              1101.0
55       Blinkit  Electric Vehicles  2196.0        2022.0              1098.0
167  Fitnessplus              AI/ML  1430.0        2022.0               715.0
28      Rupaypro        Travel-Tech  3541.0        2019.0               708.2
205    Docontime         TRAVELTECH  3529.0        2019.0               705.8
144    Groomguru             Gaming  3497.0        2019.0               699.4
382    Agriboost              Games  3507.0        2018.0               584.5
146    Curefoods          Food-Tech  2230.0        2020.0               557.5


In [22]:
# AI-Assisted: Defined a custom mapping function to process the amount column cleanly row-by-row.
def get_funding_tier(amount):
    if pd.isna(amount):
        return 'Small / Unknown'
    elif amount >= 1000:
        return 'Unicorn'
    elif amount >= 500:
        return 'Mega'
    elif amount >= 100:
        return 'Large'
    elif amount >= 10:
        return 'Mid'
    else:
        return 'Small / Unknown'

df['funding_tier'] = df['amount'].apply(lambda x: get_funding_tier(x))
print(df['funding_tier'].value_counts())

funding_tier
Small / Unknown    282
Large               53
Mid                 50
Unicorn             38
Mega                27
Name: count, dtype: int64


In [23]:
bins = [0, 2009, 2017, 3000]
labels = ['Early', 'Growth', 'Recent']
df['era'] = pd.cut(df['founded_year'], bins=bins, labels=labels)
print(df['era'].value_counts())

era
Recent    207
Growth    195
Early      25
Name: count, dtype: int64


In [24]:
# AI-Assisted: Lambda checks for NaN first. If not NaN, splits the string by comma-space and counts elements.
df['investor_count'] = df['investors'].apply(lambda x: len(str(x).split(', ')) if pd.notna(x) else 0)
print(df['investor_count'].value_counts())

investor_count
2    199
3    118
1     94
4     29
0     10
Name: count, dtype: int64


Aggregation & Grouping

In [25]:
q33_result = df.groupby('industry')['amount'].mean().dropna().sort_values(ascending=False).head(5)
print("Top 5 industries by average funding:\n", q33_result)

Top 5 industries by average funding:
 industry
Fintech       2863.500000
TRAVELTECH    1609.600000
Food-Tech     1500.250000
Logistics     1482.333333
HealthTech    1421.000000
Name: amount, dtype: float64


In [26]:
q34_result = df.groupby('city').agg({
    'startup_name': 'count',
    'amount': ['sum', 'mean']
})
q34_result.columns = ['startup_count', 'total_funding', 'avg_funding']
q34_result = q34_result.sort_values(by='startup_count', ascending=False).head(10)
print(q34_result)

           startup_count  total_funding  avg_funding
city                                                
Noida                 62        15500.0   673.913043
Kolkata               51        12786.0   639.300000
Bengaluru             49        19525.0   929.761905
Ahmedabad             44        18372.0   966.947368
Pune                  42        12797.0   710.944444
Chennai               37         9750.0   609.375000
Gurugram              36        14136.0   831.529412
Mumbai                35         7802.0   709.272727
New Delhi             33         8626.0   958.444444
Hyderabad             30         7247.0   805.222222


In [27]:
industry_stats = df.groupby('industry').agg(
    startup_count=('startup_name', 'count'),
    avg_funding=('amount', 'mean')
)
filtered_industries = industry_stats[industry_stats['startup_count'] >= 15]
top_industry = filtered_industries.sort_values(by='avg_funding', ascending=False).head(1)
print(top_industry)

            startup_count  avg_funding
industry                              
TRAVELTECH             16       1609.6


In [28]:
top_5_industries = df['industry'].value_counts().head(5).index
q36_df = df[df['industry'].isin(top_5_industries)]
crosstab_result = pd.crosstab(q36_df['industry'], q36_df['funding_tier'])
print(crosstab_result)

funding_tier  Large  Mega  Mid  Small / Unknown  Unicorn
industry                                                
AI/ML             3     1    3               12        3
PROPTECH          1     2    0                9        1
TRAVELTECH        0     3    0               11        2
logistics         2     1    2                9        0
saas              2     1    1               10        0


In [29]:
# Drop NaN amount rows first so idxmax() operates on clean comparable data
valid_amounts = df.dropna(subset=['amount'])
highest_fund_idx = valid_amounts.groupby('industry')['amount'].idxmax()
q37_result = valid_amounts.loc[highest_fund_idx, ['startup_name', 'industry', 'city', 'amount']]
Real Industry Problemsprint(q37_result)

      startup_name           industry       city  amount
257       Edusmart           AGRITECH      Noida   193.0
126  Bookmyservice              AI ML    Chennai  1443.0
406         Glowup              AI-ML     Mumbai  1441.0
322     Classroomx              AI/ML  Ahmedabad  3520.0
339      Sharechat               AIML   Gurugram   900.0
404      Beautybox          Agri Tech   Gurugram   183.0
2       Freshworks           AgriTech    Kolkata   329.0
98          Meesho         BEAUTYTECH  Bengaluru  2231.0
93       Vitalcare         BeautyTech       Pune  3522.0
411        Fetchit         E-commerce       Pune    41.0
266    Whitehat Jr          ECOMMERCE    Kolkata    98.0
316      Instaloan                 EV    Chennai   923.0
9      Mediconnect          Ecommerce      Noida   295.0
461        Postman             EdTech  Hyderabad  1404.0
175       Propmart             Edtech  New Delhi  3534.0
311       Routemap  Electric Vehicles  Bengaluru  3528.0
12             Ola  Electric-Ve

Real Industry Problems

In [30]:
target_industries = ['Fintech', 'Edtech', 'Health Tech']
emerging_players = df[
    (df['founded_year'] >= 2019) &
    (df['amount'] >= 100) &
    (df['industry'].isin(target_industries))
]
emerging_players = emerging_players.sort_values(by='amount', ascending=False)
print(emerging_players[['startup_name', 'industry', 'city', 'founded_year', 'amount']])

print("\n# Explanation: A VC analyst would care about this list because it highlights high-growth, highly capitalised companies in trending sectors that have proven rapid market traction in just a few years.")

    startup_name     industry      city  founded_year  amount
342      Voltcar       Edtech       Hyd        2021.0   916.0
417  Villagemart       Edtech  Gurugram        2021.0   896.0
130    Dermaplus  Health Tech  Gurugram        2019.0   711.0

# Explanation: A VC analyst would care about this list because it highlights high-growth, highly capitalised companies in trending sectors that have proven rapid market traction in just a few years.


In [31]:
city_stats = df.groupby('city').agg(
    startup_count=('startup_name', 'count'),
    total_funding=('amount', 'sum')
)
valid_cities = city_stats[city_stats['startup_count'] >= 20]
top_hub = valid_cities.sort_values(by='total_funding', ascending=False).head(1)
print(top_hub)

           startup_count  total_funding
city                                   
Bengaluru             49        19525.0


In [32]:
saturation = df.groupby(['city', 'industry']).size().reset_index(name='startup_count')
top_10_saturated = saturation.sort_values(by='startup_count', ascending=False).head(10)
print("Top 10 most saturated (city, industry) markets:\n", top_10_saturated)

Top 10 most saturated (city, industry) markets:
           city           industry  startup_count
254      Noida         BEAUTYTECH              6
264      Noida           FoodTech              5
287       Pune              AI/ML              4
277      Noida         TRAVELTECH              4
241  New Delhi          LOGISTICS              4
165    Kolkata              AI/ML              4
288       Pune               AIML              3
41   Bengaluru  Electric Vehicles              3
305       Pune           PROPTECH              3
50   Bengaluru        Travel-Tech              3
